# File Search

*Notebook 11*

Give your agent access to your own documents using OpenAI's built-in vector search.
<br>
<br>
**Topics:**
- What a vector store is and why it's needed
- Uploading files and creating a vector store
- Querying documents with `FileSearchTool`
- Grounding vs guessing — when the agent answers from documents vs training data
- Cleaning up the demo vector store

---

## 🔧 Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from openai import OpenAI
from agents import Agent, FileSearchTool, Runner

MODEL = "gpt-5-mini"

# OpenAI client for vector store management
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


print("✅ Ready!")

---

## 🎯 The Problem

Web search finds public information — but not your own documents. File Search gives your agent access to private files using semantic and keyword search to retrieve relevant chunks as grounded context.

---

## 📚 Part 1: What Is a Vector Store?

File Search uses semantic and keyword search — not plain text matching. Your documents are chunked, converted into vectors, and stored. At query time the agent finds the closest matching chunks and injects them into context as grounded evidence.

This pattern is called **RAG** — Retrieval-Augmented Generation. In practice, that means the agent first **retrieves** relevant document chunks, then **generates** an answer using those chunks as evidence. A *vector* is a numerical representation of meaning — similar content ends up with similar numbers, letting the system find relevant passages even when the exact words don't match. File Search uses semantic search (matching by meaning) plus keyword search as a fallback, and handles all of it automatically. Note: File Search uses vector search under the hood — we build our own in Lesson 21: Vector Memory.

### 💡 Cost Note

This demo should cost very little, but vector stores persist until deleted — run the cleanup cell at the end of the notebook when you're done.

---

## 📄 Part 2: Create a Sample Document

We'll create a sample product FAQ document to use as our knowledge base.

In [ ]:
# Create a sample document to upload
sample_doc = """# TechGadget Pro — Product FAQ

## What is TechGadget Pro?
TechGadget Pro is a smart productivity device that combines a noise-cancelling speaker,
AI assistant, and wireless charger in one compact unit. It connects via Bluetooth 5.3
and has a 12-hour battery life.

## What devices are compatible?
TechGadget Pro is compatible with iOS 15+, Android 11+, Windows 10+, and macOS 12+.
It supports up to 3 simultaneous Bluetooth connections.

## How do I reset TechGadget Pro?
To perform a factory reset: hold the power button and volume down button simultaneously
for 10 seconds until the LED flashes red three times. All paired devices will be cleared.

## What is the warranty policy?
TechGadget Pro comes with a 2-year limited warranty covering manufacturing defects.
Physical damage, water damage, and unauthorized modifications are not covered.
Contact support@techgadgetpro.com to initiate a warranty claim.

## How do I update the firmware?
Open the TechGadget app, navigate to Settings > Device > Firmware Update.
Updates download automatically over Wi-Fi and install when the device is charging.
Do not power off during an update.

## What is the return policy?
We accept returns within 30 days of purchase for any reason. The device must be in
original condition with all accessories. Refunds are processed within 5-7 business days.
"""

# Save to a local file
doc_path = Path("techgadget_faq.txt")
doc_path.write_text(sample_doc)

# --------------------------------------------------------------
print(f"✅ Sample document created: {doc_path}")
print(f"   Size: {len(sample_doc)} characters")

---

## ☁️ Part 3: Upload and Create a Vector Store

Vector store management uses the `openai` client directly — not the agents SDK. This is a one-time setup step. In a real application, you'd run this setup code once, save the resulting `vector_store.id`, and reuse that ID in your agent code without recreating the store on every run. Vector stores act as collections — you can create multiple stores to organize files by topic, project, or access level.

In [ ]:
print("Creating vector store...\n")

# Step 1: Create the vector store
vector_store = client.vector_stores.create(name="TechGadget FAQ")
print(f"✅ Vector store created: {vector_store.id}")

# Step 2: Upload file and poll until processing is complete
with open(doc_path, "rb") as file_handle:
    file_batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id=vector_store.id,
        files=[file_handle]
    )

print(f"✅ File uploaded and processed")
print(f"   Status: {file_batch.status}")
print(f"   Files: {file_batch.file_counts.completed} completed")

### 💡 Why This Works

`upload_and_poll` uploads the file, triggers processing, and waits until the vector store is ready — all in one call. The agent can't query files that are still processing.

---

## 🔍 Part 4: Query with FileSearchTool

Now that the document is indexed, here's the key moment: we'll attach that store to an agent and watch it answer from the document instead of guessing from model knowledge.

Now connect the vector store to an agent using `FileSearchTool`. In your own project, the pattern is the same: upload your documents to a vector store, pass that store's ID into `FileSearchTool`, and rewrite the agent instructions for your domain.

In [ ]:
instructions = (
    "You are a customer support agent for TechGadget Pro.\n"
    "Answer questions using only information from the provided documents.\n"
    "If the answer is not in the documents, say so clearly."
)

agent = Agent(
    name="SupportAgent",
    instructions=instructions,
    model=MODEL,
    tools=[FileSearchTool(
        vector_store_ids=[vector_store.id],
        max_num_results=3
    )]
)

# --------------------------------------------------------------
print("✅ Support agent ready")

#### Test: Warranty Question

In [ ]:
print("🔍 FILE SEARCH DEMO")
print("=" * 60)

result = await Runner.run(agent, input="What is the warranty policy?")

print(f"Q: What is the warranty policy?")
print(f"A: {result.final_output}")

#### Test: Reset Instructions

In [ ]:
result = await Runner.run(agent, input="How do I factory reset the device?")

print(f"Q: How do I factory reset the device?")
print(f"A: {result.final_output}")

#### Test: Question Not in Documents

In [ ]:
result = await Runner.run(agent, input="What color options are available?")

print(f"Q: What color options are available?")
print(f"A: {result.final_output}")
print("=" * 60)

### 💡 Why This Works

The agent retrieves only the chunks relevant to each question — it doesn't load the entire document into context. This makes File Search efficient even with very large knowledge bases.

Notice the third test — the agent refused to invent an answer. Without grounding, the model would likely guess plausible-sounding colors. With grounding, it can only answer from what was actually retrieved — which matters most when the agent is answering about private or high-stakes content.

`max_num_results=3` caps how many document chunks the agent retrieves per query — keep it small to control context size and cost. `vector_store_ids` accepts a list, so a single agent can search across multiple knowledge bases at once — for example, a product FAQ and a user manual in the same turn.

⚠️ **Security note:** In a real application, documents often come from user uploads or external pipelines — that's why retrieved chunks are treated as untrusted input and can enable prompt injection (see Lesson 23). Treat retrieved content the same way you'd treat user input; don't pass it directly into system instructions without validation.

---

In [ ]:
# Clean up local sample files before exercises
for file in ["techgadget_faq.txt"]:
    path = Path(file)
    if path.exists():
        path.unlink()
        print(f"✅ Local file deleted: {file}")

---

## 💪 Practice Exercises

### Exercise 1: Add a Second Document

*Covers: vector stores — extending a knowledge base*

Extend the knowledge base with a second document and verify the agent can answer questions from both.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Add a Second Document
# --------------------------------------------------------------
# Objective: Add a shipping policy document to the existing vector store.

shipping_doc = """# TechGadget Pro — Shipping Policy

## Standard Shipping
Standard shipping takes 5-7 business days and costs $4.99.
Free standard shipping on orders over $50.

## Express Shipping
Express shipping takes 2 business days and costs $12.99.

## International Shipping
We ship to 45 countries. International orders take 10-14 business days.
Customs fees are the responsibility of the customer.
"""

# TODO 1: Save shipping_doc to a file called shipping_policy.txt

# TODO 2: Upload it to the existing vector_store using
#          client.vector_stores.file_batches.upload_and_poll()
#          Hint: use vector_store_id=vector_store.id

# TODO 3: Run the agent with: "How much does express shipping cost?"
#          and print the result

# TODO 4: Run a second query that requires the FAQ document (not the shipping doc)
#          to confirm both documents are accessible from the same vector store

# --- Write your code below this line ---

### Exercise 2: Build a New Knowledge Base

*Covers: vector stores — building and querying from scratch*

Create a new vector store with a different document and query it with a new agent.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Build a New Knowledge Base

# --------------------------------------------------------------
# Objective: Create a separate vector store and query it independently.

return_policy_doc = """# TechGadget Pro — Return Policy

## Standard Returns
Returns accepted within 30 days of purchase.
Items must be in original condition with all accessories included.

## Damaged Items
If your item arrived damaged, contact support within 7 days with photos.
We will send a replacement at no additional cost.

## Refund Timeline
Refunds are processed within 5-7 business days after we receive the item.
"""

# TODO 1: Save return_policy_doc to return_policy.txt

# TODO 2: Create a new vector store (give it a different name)
#          Upload return_policy.txt to it

# TODO 3: Create a new agent with the new vector store
#          Use different instructions — e.g. a returns specialist

# TODO 4: Run the agent with: "How long do I have to return a damaged item?"
#          Confirm it answers from the return policy, not the FAQ

# TODO 5: Delete the new vector store when done

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**Vector stores enable RAG:**
- Files are chunked, converted into vectors, and stored for semantic and keyword search
- The agent retrieves only relevant chunks — not the entire document
- OpenAI manages the infrastructure; you just upload and query
<br>
<br>
**Two-step setup:**
- Create and populate the vector store with the `openai` client
- Connect it to the agent with `FileSearchTool(vector_store_ids=[...])`
<br>
<br>
**Always clean up:**
- Vector stores persist and accrue storage costs until deleted
- Delete with `client.vector_stores.delete(vector_store.id)` when done

---

## 📍 Next Step

**Notebook 12: Code Interpreter**  

Let your agent write and execute Python code in a sandboxed environment to analyze data and generate outputs.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-11-file-search)

---

#### Cleanup: Delete the Vector Store

Vector stores persist on OpenAI's servers until deleted. Run this cell when you're done to avoid ongoing storage charges. For demos, delete the store when you're done; in a real app, you'd usually keep the store and reuse its ID until the underlying documents change.

In [ ]:
client.vector_stores.delete(vector_store.id)
print(f"✅ Vector store deleted: {vector_store.id}")

---